# Notebook 03 — Model Development

**Project:** Blood Donation Prediction System  
**Authors:** Mihret Alemayehu, Abebech Nega  
**Institution:** Debre Berhan University  
**Instructor:** Petros Abebe  

---

## Objective
Train a **Random Forest Classifier** on the preprocessed blood donation
dataset and save the trained model and preprocessing pipeline to disk:
1. Load cleaned data from `data/processed/`
2. Scale features using StandardScaler
3. 80/20 train/test split (stratified, `random_state=42`)
4. Train Random Forest (`n_estimators=100`)
5. 5-fold cross-validation
6. Save `model.pkl`, `preprocessing.pkl`, and `model_metadata.json`


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import os
import sys
import pickle
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

warnings.filterwarnings('ignore')

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.features.builder import FeatureBuilder
from src.models.trainer import ModelTrainer
from src.visualization.plotter import Plotter

sns.set_theme(style='whitegrid', font_scale=1.1)
%matplotlib inline
print('Imports successful.')

---
## 1. Load Processed Data


In [ ]:
processed_path = os.path.join(PROJECT_ROOT, 'data', 'processed', 'blood_donation_clean.csv')

if not os.path.exists(processed_path):
    raise FileNotFoundError(
        f'Processed file not found: {processed_path}\n'
        'Please run Notebook 02 (02_preprocessing.ipynb) first.'
    )

df_clean = pd.read_csv(processed_path)
print(f'Loaded cleaned dataset: {df_clean.shape}')
df_clean.head()

---
## 2. Feature / Target Separation and Scaling


In [ ]:
target_col = 'Target'

# Separate features from the label
y = df_clean[target_col].copy()
X = df_clean.drop(columns=[target_col])

print(f'Feature matrix X: {X.shape}')
print(f'Target series  y: {y.shape}')
print(f'\nClass distribution:\n{y.value_counts()}')
print(f'\nClass balance: {y.value_counts(normalize=True).round(3).to_dict()}')

In [ ]:
# Apply StandardScaler to the feature columns
# Scaling ensures features with large ranges (e.g., Monetary) don't
# dominate distance-based computations in the ensemble.
scaler = StandardScaler()
X_scaled_array = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled_array, columns=X.columns)

print('Features scaled with StandardScaler.')
print(X_scaled.describe().T[['mean', 'std']].round(4))

---
## 3. Train / Test Split — 80% / 20%


In [ ]:
# Stratified split preserves the class proportions in both subsets.
# random_state=42 ensures reproducibility across runs.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f'Training set : {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X_scaled)*100:.1f}%%)')
print(f'Test set     : {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X_scaled)*100:.1f}%%)')
print(f'\nTrain class distribution: {y_train.value_counts().to_dict()}')
print(f'Test  class distribution: {y_test.value_counts().to_dict()}')

---
## 4. Train the Random Forest Classifier


In [ ]:
# Initialize the Random Forest with course-specified hyper-parameters:
#   n_estimators = 100  (build 100 decision trees)
#   random_state = 42   (reproducibility)
#   class_weight = 'balanced'  (handle slight class imbalance)

rf_model = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced',
)

print('Training Random Forest Classifier...')
rf_model.fit(X_train, y_train)

train_accuracy = rf_model.score(X_train, y_train)
print(f'\nTraining Accuracy: {train_accuracy:.4f} ({train_accuracy*100:.2f}%%)')

---
## 5. Cross-Validation (5-Fold)


In [ ]:
# Cross-validation gives a more reliable estimate of generalization
# performance by training on 5 different data splits.
cv_scores = cross_val_score(rf_model, X_scaled, y, cv=5, scoring='accuracy', n_jobs=-1)

print('=== 5-Fold Cross-Validation Results ===')
for i, score in enumerate(cv_scores, 1):
    print(f'  Fold {i}: {score:.4f}')
print(f'\n  Mean   : {cv_scores.mean():.4f}')
print(f'  Std Dev: {cv_scores.std():.4f}')

---
## 6. Feature Importances


In [ ]:
# Gini importance — average reduction in impurity across all trees
feature_importances = pd.DataFrame({
    'feature'   : X.columns,
    'importance': rf_model.feature_importances_,
}).sort_values('importance', ascending=False).reset_index(drop=True)

print('=== Feature Importances (Top 10) ===')
print(feature_importances.head(10).to_string(index=False))

# Save the importance chart
plotter = Plotter(save_dir=os.path.join(PROJECT_ROOT, 'reports', 'figures'))
plotter.feature_importance_plot(feature_importances, top_n=len(feature_importances))
print('\nFeature importance chart saved.')

---
## 7. Save Model and Preprocessing Pipeline


In [ ]:
models_dir = os.path.join(PROJECT_ROOT, 'models')
os.makedirs(models_dir, exist_ok=True)

# ── Save model.pkl ────────────────────────────────────────────────────────────
model_path = os.path.join(models_dir, 'model.pkl')
with open(model_path, 'wb') as f:
    pickle.dump(rf_model, f)
print(f'Trained model saved to : {model_path}')

# ── Save preprocessing.pkl (the fitted StandardScaler) ───────────────────────
preprocess_path = os.path.join(models_dir, 'preprocessing.pkl')
with open(preprocess_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'Preprocessing pipeline saved to: {preprocess_path}')

In [ ]:
# ── Save model_metadata.json ──────────────────────────────────────────────────
from datetime import datetime

metadata = {
    'project'          : 'Blood Donation Prediction System',
    'authors'          : ['Mihret Alemayehu', 'Abebech Nega'],
    'institution'      : 'Debre Berhan University',
    'instructor'       : 'Petros Abebe',
    'algorithm'        : 'Random Forest Classifier',
    'hyperparameters'  : {
        'n_estimators'  : 100,
        'random_state'  : 42,
        'class_weight'  : 'balanced',
        'n_jobs'        : -1,
    },
    'split'            : {'test_size': 0.20, 'train_size': 0.80, 'stratified': True},
    'train_samples'    : int(X_train.shape[0]),
    'test_samples'     : int(X_test.shape[0]),
    'features'         : X.columns.tolist(),
    'n_features'       : int(X.shape[1]),
    'cv_mean_accuracy' : round(float(cv_scores.mean()), 4),
    'cv_std_accuracy'  : round(float(cv_scores.std()), 4),
    'train_accuracy'   : round(float(train_accuracy), 4),
    'saved_at'         : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
}

meta_path = os.path.join(models_dir, 'model_metadata.json')
with open(meta_path, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f'Model metadata saved to: {meta_path}')
print(json.dumps(metadata, indent=2))

---
## 8. Summary

| Item | Value |
|---|---|
| Algorithm | Random Forest Classifier |
| Trees (n_estimators) | 100 |
| Train / Test Split | 80% / 20% |
| 5-Fold CV Accuracy | see above |
| Saved artifacts | model.pkl, preprocessing.pkl, model_metadata.json |

> Proceed to **Notebook 04** for model evaluation and report generation.


---
## 9. Baseline Model — Logistic Regression

A **baseline model** is a simple benchmark that the main model must outperform.
We use **Logistic Regression** as the baseline — it is interpretable,
fast, and commonly used for binary classification.


In [ ]:
# ── Baseline: Logistic Regression ────────────────────────────────────────────
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, precision_score, recall_score

# Dummy baseline (always predicts majority class)
dummy = DummyClassifier(strategy='most_frequent', random_state=42)
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
dummy_acc  = accuracy_score(y_test, dummy_pred)

# Logistic Regression baseline
lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train, y_train)
lr_pred  = lr.predict(X_test)
lr_proba = lr.predict_proba(X_test)[:, 1]
lr_acc   = accuracy_score(y_test, lr_pred)
lr_f1    = f1_score(y_test, lr_pred, zero_division=0)
lr_auc   = roc_auc_score(y_test, lr_proba)
lr_prec  = precision_score(y_test, lr_pred, zero_division=0)
lr_rec   = recall_score(y_test, lr_pred, zero_division=0)

print('=== Dummy Baseline ===')
print(f'  Accuracy : {dummy_acc:.4f}')

print('\n=== Logistic Regression Baseline ===')
print(f'  Accuracy  : {lr_acc:.4f}')
print(f'  F1 Score  : {lr_f1:.4f}')
print(f'  ROC-AUC   : {lr_auc:.4f}')
print(f'  Precision : {lr_prec:.4f}')
print(f'  Recall    : {lr_rec:.4f}')


---
## 10. Model Comparison — Random Forest vs Logistic Regression vs Dummy

We compare all three models on the **same held-out test set** using identical
evaluation metrics. This satisfies the instruction requirement for model comparison.


In [ ]:
# ── Model Comparison Table ────────────────────────────────────────────────────
rf_pred  = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_acc   = accuracy_score(y_test, rf_pred)
rf_f1    = f1_score(y_test, rf_pred, zero_division=0)
rf_auc   = roc_auc_score(y_test, rf_proba)
rf_prec  = precision_score(y_test, rf_pred, zero_division=0)
rf_rec   = recall_score(y_test, rf_pred, zero_division=0)

comparison_df = pd.DataFrame({
    'Model'    : ['Dummy (majority)', 'Logistic Regression', 'Random Forest (final)'],
    'Accuracy' : [round(dummy_acc, 4), round(lr_acc, 4), round(rf_acc, 4)],
    'F1 Score' : ['-',                 round(lr_f1, 4),  round(rf_f1, 4)],
    'ROC-AUC'  : ['-',                 round(lr_auc, 4), round(rf_auc, 4)],
    'Precision': ['-',                 round(lr_prec,4), round(rf_prec,4)],
    'Recall'   : ['-',                 round(lr_rec, 4), round(rf_rec, 4)],
    'Selected' : ['❌', '—', '✅'],
})

print('=== Model Comparison ===')
print(comparison_df.to_string(index=False))


In [ ]:
# ── Bar chart comparison ──────────────────────────────────────────────────────
metrics  = ['Accuracy', 'F1 Score', 'ROC-AUC', 'Precision', 'Recall']
lr_vals  = [lr_acc,   lr_f1,   lr_auc,   lr_prec,   lr_rec]
rf_vals  = [rf_acc,   rf_f1,   rf_auc,   rf_prec,   rf_rec]

x = range(len(metrics))
width = 0.32

fig, ax = plt.subplots(figsize=(11, 5))
fig.patch.set_facecolor('#1E293B')
ax.set_facecolor('#1E293B')

bars1 = ax.bar([i - width/2 for i in x], lr_vals, width,
               label='Logistic Regression', color='#3498DB', alpha=0.85, edgecolor='#0F172A')
bars2 = ax.bar([i + width/2 for i in x], rf_vals, width,
               label='Random Forest', color='#E74C3C', alpha=0.85, edgecolor='#0F172A')

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{bar.get_height():.3f}', ha='center', va='bottom',
                fontsize=8, color='white', fontweight='bold')

ax.set_xticks(list(x))
ax.set_xticklabels(metrics, color='white', fontsize=11)
ax.set_ylabel('Score', color='#94A3B8')
ax.set_ylim(0, 1.15)
ax.set_title('Model Comparison: Random Forest vs Logistic Regression',
             color='white', fontsize=13)
ax.legend(fontsize=10, facecolor='#334155', labelcolor='white')
ax.tick_params(colors='white')
for sp in ax.spines.values(): sp.set_color('#334155')
ax.grid(axis='y', color='#334155', alpha=0.4, linestyle='--')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'reports', 'figures', 'model_comparison.png'),
            dpi=120, bbox_inches='tight', facecolor='#1E293B')
plt.show()
print('Figure saved: reports/figures/model_comparison.png')
